In [2]:
# ============================================================
# Notebook 04: Logistic Regression Baseline
# AI Financial Risk Intelligence Platform
# ============================================================

from pathlib import Path
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# ----------------------------
# Load dataset
# ----------------------------

PROJECT_ROOT = Path(r"C:\Users\user\machine-learning-models-practical\AIML Engineer Training Series\AI_Financial_Risk_Intelligence_Platform")

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "german.data"

column_names = [
    "status", "duration", "credit_history", "purpose", "credit_amount",
    "savings", "employment_duration", "installment_rate", "personal_status_sex",
    "other_debtors", "present_residence", "property", "age",
    "other_installment_plans", "housing", "existing_credits", "job",
    "people_liable", "telephone", "foreign_worker", "target"
]

df = pd.read_csv(
    DATA_PATH,
    sep=r"\s+",
    header=None,
    names=column_names
)

df["target"] = df["target"].map({1: 0, 2: 1})

# ----------------------------
# Features and target
# ----------------------------

X = df.drop(columns="target")
y = df["target"]

# ----------------------------
# Train/Test split
# ----------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ----------------------------
# Feature types
# ----------------------------

categorical_cols = X_train.select_dtypes(include="object").columns.tolist()
numerical_cols = X_train.select_dtypes(exclude="object").columns.tolist()

# ----------------------------
# Preprocessing
# ----------------------------

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

# ----------------------------
# Pipeline
# ----------------------------

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ]
)

print("Pipeline created successfully")

Pipeline created successfully


In [3]:
pipeline.fit(X_train, y_train)

print("Logistic Regression pipeline trained successfully!")

Logistic Regression pipeline trained successfully!


In [4]:
# ============================================================
# Predictions
# ============================================================

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print("First 10 predicted labels:")
print(y_pred[:10])

print("\nFirst 10 predicted probabilities:")
print(y_prob[:10])

First 10 predicted labels:
[0 0 1 0 0 0 0 0 0 0]

First 10 predicted probabilities:
[0.22680794 0.10145826 0.61876846 0.49335708 0.15016187 0.11255578
 0.21671203 0.11332271 0.48857598 0.15060426]


In [5]:
# ============================================================
# Evaluation Metrics
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")

Accuracy : 0.7800
Precision: 0.6667
Recall   : 0.5333
F1 Score : 0.5926
ROC-AUC  : 0.8040


In [6]:
# ============================================================
# Confusion Matrix
# ============================================================

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[124  16]
 [ 28  32]]


## Confusion Matrix Interpretation

The Logistic Regression model correctly classified **124 good-credit applicants** and **32 bad-credit applicants**.

However, it incorrectly approved **28 risky borrowers** (false negatives) and rejected **16 safe borrowers** (false positives).

From a banking perspective, **false negatives are more expensive** because they can result in loan defaults and financial losses. Therefore, future iterations of the model should focus on improving **recall** while maintaining a reasonable level of precision.

This suggests that threshold tuning or class-weight adjustment may improve business performance even if overall accuracy changes only slightly.


In [8]:
# ============================================================
# Logistic Regression Coefficients
# ============================================================

feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()

coefficients = pipeline.named_steps["model"].coef_[0]

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

coef_df = coef_df.sort_values(by="Coefficient", ascending=False)

print("Top 15 Risk-Increasing Features")
display(coef_df.head(15))

print("\nTop 15 Risk-Decreasing Features")
display(coef_df.tail(15))

Top 15 Risk-Increasing Features


,Feature,Coefficient
23,cat__purpose_A46,0.901684
46,cat__property_A124,0.713710
7,cat__status_A11,0.657834
26,cat__savings_A61,0.640212
59,cat__foreign_worker_A201,0.567781
50,cat__housing_A151,0.561244
11,cat__credit_history_A30,0.540755
16,cat__purpose_A40,0.516158
22,cat__purpose_A45,0.442572
36,cat__personal_status_sex_A91,0.437304



Top 15 Risk-Decreasing Features


,Feature,Coefficient
53,cat__job_A171,-0.225599
43,cat__property_A121,-0.371877
24,cat__purpose_A48,-0.399603
49,cat__other_installment_plans_A143,-0.410521
30,cat__savings_A65,-0.429549
42,cat__other_debtors_A103,-0.451195
18,cat__purpose_A410,-0.481805
38,cat__personal_status_sex_A93,-0.501379
34,cat__employment_duration_A74,-0.539283
60,cat__foreign_worker_A202,-0.581531


## Logistic Regression Coefficient Interpretation

The Logistic Regression model identified several financially meaningful predictors of credit risk.

### Risk-Increasing Features

* **status_A11**
* **savings_A61**
* **credit_history_A30**
* **duration**
* **installment_rate**

These features increase the probability of belonging to the bad-credit class.

### Risk-Decreasing Features

* **status_A14**
* **credit_history_A34**
* **savings_A64**
* **housing_A153**

These features reduce the probability of default.

The coefficient signs were consistent with the patterns observed during exploratory data analysis, indicating that the model successfully captured meaningful financial relationships present in the dataset.
